In [1]:
import os
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.messages import SystemMessage
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain.messages import AIMessage,HumanMessage,SystemMessage
import requests
import smtplib
from email.message import EmailMessage
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()


C:\Users\Azrap\AppData\Local\Temp\ipykernel_11536\2714394287.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [3]:
resume_path=r"C:\Users\Azrap\Downloads\resume.pdf"

In [4]:
linkedin_url="https://www.linkedin.com/in/azra-patvi-5ba33a31b/"
github_url="https://github.com/azrapatvi/"

In [5]:
loader=PyPDFLoader(resume_path)
resume_text="\n".join(p.page_content for p in loader.load())

In [6]:
class State(TypedDict):
    messages:Annotated[list,add_messages]
    details:str
    recruiter_email:str
    approved:bool
    drafted_email_body:str
    drafted_email_subject:str
    company_name:str
    job_role: str
    resume_text:str

In [7]:

from langchain_core.prompts import ChatPromptTemplate


extract_prompt = ChatPromptTemplate.from_template("""
Extract the following information from the user message:

1. Candidate details
2. Recruiter email address
3. Job role
4. Company name

User message:
{user_message}

Return exactly in this format:
Details: <candidate details>
Recruiter Email: <email address>
Job Role: <job role>
Company Name: <company_name>

If something is missing, write Missing.
""")


draft_email_prompt = ChatPromptTemplate.from_template("""
You are a professional job application email writer.

Write a personalized, clear, and convincing application email
using the candidate's resume and details.

Candidate Resume:
{resume_text}

LinkedIn Profile:
{linkedin_url}

GitHub Profile:
{github_url}

Candidate Details:
{details}

Recruiter Email:
{recruiter_email}

Job Role:
{job_role}

Company Name:
{company_name}


EMAIL STRUCTURE

1. Opening Paragraph:
- Mention the job role and company name.
- Briefly introduce the candidate.
- Express interest in the position.

2. Relevant Professional Experience:
- This section is MANDATORY.
- Mention relevant companies and job titles.
- Explain actual responsibilities and technologies used.
- Include measurable achievements when available.
- Prioritize experience relevant to the target role.
- Do not skip this section, even if the candidate has projects.

3. Relevant Project Experience:
- This section is MANDATORY and must come after professional experience.
- Select 1-3 projects relevant to the target job.
- Mention each project's name, purpose, technologies, and implementation.
- Include specific results only when supported by the resume.
- Do not combine all projects into one vague sentence.
- Do not describe every project in the resume.

4. Closing Paragraph:
- Explain briefly how the candidate's skills can contribute to the company.
- Mention that the resume is attached.
- Express interest in discussing the opportunity.

5. Signature:
- End with "Regards," followed by the candidate's actual name.
- Include the candidate's email address if available in the resume.
- Include the COMPLETE LinkedIn URL provided above.
- Include the COMPLETE GitHub URL provided above.
- Never write only "LinkedIn" or "GitHub" without their URLs.
- Never invent or modify contact details.

WRITING RULES

- Aim for 180-250 words, excluding the subject and signature.
- Never exceed 300 words.
- Use short paragraphs and clear section headings.
- Mention professional experience FIRST, then projects.
- Use a professional, natural, confident tone.
- Do not reproduce the entire resume.
- Do not invent experience, responsibilities, technologies, skills,
  projects, achievements, metrics, or qualifications.
- Use only information supported by the resume or candidate details.
- Use "Dear Hiring Manager" unless the recruiter's name is reliably known.
- Do not invent the recruiter's name.
- Do not include placeholder information.
- Do not use Markdown formatting markers such as **.
- Keep the subject concise and specific to the role and company.
- Do not invent, guess, or modify any email address or profile URL.



Return the email in exactly this format:

Subject: <email_subject>

Body:
Dear Hiring Manager,

<opening paragraph>

Relevant Professional Experience:
<relevant experience with specific responsibilities>

Relevant Project Experience:
<1-3 selected projects with specific details>

<closing paragraph>

Regards,
<candidate name>
Email: <candidate email if available>
LinkedIn: <complete LinkedIn URL>
GitHub: <complete GitHub URL>
""")


In [8]:
sender_email=os.getenv("SENDER_EMAIL")
app_password = os.getenv('GMAIL_APP_PASSWORD')

In [9]:
graph_builder=StateGraph(State)



def extract_info(state:State):
    """to extract the information from the messages send by the candidate"""
    user_message = state["messages"][-1].content

    response = (extract_prompt | llm).invoke({
        "user_message": user_message
    })

    # Parse the LLM response
    lines = response.content.splitlines()

    details = ""
    recruiter_email = ""
    job_role = ""
    company_name=""

    for line in lines:
        if line.startswith("Details:"):
            details = line.split(":", 1)[1].strip()

        elif line.startswith("Recruiter Email:"):
            recruiter_email = line.split(":", 1)[1].strip()

        elif line.startswith("Job Role:"):
            job_role = line.split(":", 1)[1].strip()
        elif line.startswith("Company Name"):
            company_name=line.split(":", 1)[1].strip()

    return {
        "details": details,
        "recruiter_email": recruiter_email,
        "job_role": job_role,
        "company_name":company_name,
        
    }



def draft_email(state: State):
    """Generate and separate the email subject and body."""

    res = (draft_email_prompt | llm).invoke({
        "resume_text": state["resume_text"],
        "details": state["details"],
        "recruiter_email": state["recruiter_email"],
        "job_role": state["job_role"],
        "company_name": state["company_name"],
         "linkedin_url": linkedin_url,
    "github_url": github_url
    })

    email_text = res.content.strip()

    # Remove Markdown formatting around the labels
    email_text = email_text.replace("**Subject:**", "Subject:")
    email_text = email_text.replace("**Body:**", "Body:")
    email_text = email_text.replace("**Subject**:", "Subject:")
    email_text = email_text.replace("**Body**:", "Body:")

    lines = email_text.splitlines()

    subject = f"Application for {state['job_role']} at {state['company_name']}"
    body_lines = []
    body_started = False

    for line in lines:
        clean_line = line.strip()

        if clean_line.lower().startswith("subject:"):
            subject = clean_line.split(":", 1)[1].strip()

        elif clean_line.lower() == "body:":
            body_started = True

        elif body_started:
            body_lines.append(line)

    body = "\n".join(body_lines).strip()

    return {
        "messages": [res],
        "drafted_email_subject": subject,
        "drafted_email_body": body,
    }


def human_approval(state: State):
    decision = interrupt({
        "type": "approval",
        "recruiter_email": state["recruiter_email"],
        "subject": state["drafted_email_subject"],
        "body": state["drafted_email_body"],
        "instruction": "Are you sure you want to send this email? (yes/no)"
    })

    approved = (
        str(decision.get("approved", "no")).strip().lower()
        in ["yes", "y", "true"]
    )

    if not approved:
        return {
            "approved": False,
            "messages": [
                AIMessage(content="Email rejected. The email was NOT sent.")
            ]
        }

    return {
        "approved": True,
        "messages": [
            AIMessage(content="Email approved. Sending email...")
        ]
    }

def check_approval(state: State):
    if state["approved"]:
        return "send_email"
    else:
        return "end"

import smtplib
from email.message import EmailMessage
from html import escape

def send_email(state: State):
    msg = EmailMessage()

    msg["Subject"] = state["drafted_email_subject"]
    msg["From"] = sender_email
    msg["To"] = state["recruiter_email"]

    body = state["drafted_email_body"]

    # Convert Markdown-style bold into HTML bold
    import re

    safe_body = escape(body)
    safe_body = re.sub(
        r"\*\*(.*?)\*\*",
        r"<strong>\1</strong>",
        safe_body,
        flags=re.DOTALL
    )

    # Convert new lines into HTML line breaks
    html_body = safe_body.replace("\n", "<br>\n")

    # Plain-text fallback
    msg.set_content(body)

    # HTML version: Gmail will render bold text
    msg.add_alternative(
        f"""
        <html>
        <body style="font-family: Arial, sans-serif;
                     font-size: 14px; line-height: 1.6;">
            {html_body}
        </body>
        </html>
        """,
        subtype="html"
    )

    # Attach the original resume PDF
    with open(resume_path, "rb") as f:
        msg.add_attachment(
            f.read(),
            maintype="application",
            subtype="pdf",
            filename="Azra_Patvi_Resume.pdf"
        )

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(sender_email, app_password)
            server.send_message(msg)

    except Exception as e:
        return {
            "messages": [
                AIMessage(content=f"Failed to send email: {e}")
            ]
        }

    return {
        "messages": [
            AIMessage(
                content=f"Email sent successfully to {state['recruiter_email']}"
            )
        ]
    }

    

graph_builder.add_node('extract_info',extract_info)
graph_builder.add_node('draft_email',draft_email)
graph_builder.add_node('human_approval',human_approval)
graph_builder.add_node('send_email',send_email)


graph_builder.add_edge(START,'extract_info')
graph_builder.add_edge('extract_info','draft_email')
graph_builder.add_edge('draft_email','human_approval')
graph_builder.add_conditional_edges('human_approval',
                                    check_approval,
                                    {
                                        "send_email":"send_email",
                                        "end":END
                                    })
graph_builder.add_edge('send_email',END)

graph = graph_builder.compile(
    checkpointer=InMemorySaver()
)



In [10]:


config = {
    "configurable": {
        "thread_id": "email_001"
    }
}

result = graph.invoke(
    {
        "messages": """
        My name is Azra Patvi.
        I am an AI and Data Science graduate.
        I have skills in Python and Machine Learning.
        I want to apply for an ML Engineer position at cognizant.
        in there comapny and this is the recruiters email id azrapatvi1234@gmail.com
        """,
        "resume_text":resume_text
        
    },
    config=config
)

result

{'messages': [HumanMessage(content='\n        My name is Azra Patvi.\n        I am an AI and Data Science graduate.\n        I have skills in Python and Machine Learning.\n        I want to apply for an ML Engineer position at cognizant.\n        in there comapny and this is the recruiters email id azrapatvi1234@gmail.com\n        ', additional_kwargs={}, response_metadata={}, id='a2553ef3-3679-4af4-a69b-3044ee9e769f'),
  AIMessage(content='Subject: Application for ML Engineer – Cognizant  \n\nBody:  \nDear Hiring Manager,\n\nI am writing to apply for the ML Engineer position at Cognizant. I am currently completing a B.Tech in Artificial Intelligence and Data Science and have hands‑on experience building end‑to‑end machine‑learning solutions. I am eager to bring my strong Python, model‑building, and deployment skills to your innovative team.\n\n**Relevant Professional Experience:**  \n*Data Scientist Intern – Vinayak IT Solutions (Feb\u202f2026 – May\u202f2026)* – Designed and deployed

In [11]:
final_res=graph.invoke(
    Command(
        resume={"approved":"no"}
    ),config=config
)
final_res

{'messages': [HumanMessage(content='\n        My name is Azra Patvi.\n        I am an AI and Data Science graduate.\n        I have skills in Python and Machine Learning.\n        I want to apply for an ML Engineer position at cognizant.\n        in there comapny and this is the recruiters email id azrapatvi1234@gmail.com\n        ', additional_kwargs={}, response_metadata={}, id='a2553ef3-3679-4af4-a69b-3044ee9e769f'),
  AIMessage(content='Subject: Application for ML Engineer – Cognizant  \n\nBody:  \nDear Hiring Manager,\n\nI am writing to apply for the ML Engineer position at Cognizant. I am currently completing a B.Tech in Artificial Intelligence and Data Science and have hands‑on experience building end‑to‑end machine‑learning solutions. I am eager to bring my strong Python, model‑building, and deployment skills to your innovative team.\n\n**Relevant Professional Experience:**  \n*Data Scientist Intern – Vinayak IT Solutions (Feb\u202f2026 – May\u202f2026)* – Designed and deployed

In [12]:
config = {"configurable": {"thread_id": "email_002"}}

result = graph.invoke(
    {
        "messages": """
        My name is Azra Patvi.
        I am an AI and Data Science graduate.
        I have skills in Python and Machine Learning.
        I want to apply for an ML Engineer position at Cognizant.
        The recruiter's email id is azrapatvi1234@gmail.com
        """,
        "resume_text":resume_text
    },
    config=config
)

In [13]:
result

{'messages': [HumanMessage(content="\n        My name is Azra Patvi.\n        I am an AI and Data Science graduate.\n        I have skills in Python and Machine Learning.\n        I want to apply for an ML Engineer position at Cognizant.\n        The recruiter's email id is azrapatvi1234@gmail.com\n        ", additional_kwargs={}, response_metadata={}, id='db1cdc6d-cf04-40fc-ac0d-f4bdbce796fb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to craft email 180-250 words, include sections. Ensure professional experience first. Provide relevant experience: Data Scientist Intern at Vinayak IT Solutions, Data Science & Analytics Intern at AMDOX Technologies, Deloitte virtual simulation maybe not needed but can mention. Must be mandatory. Then project experience: choose 1-3 projects relevant to ML Engineer: VehicleAI (price prediction), Chat With Your Data (AI-powered EDA), AI-Powered Interview Simulation System (FastAPI). Provide details. Ensure word count 180-250

In [14]:
final_res = graph.invoke(Command(resume={"approved": "yes"}), config=config)
final_res['messages'][-1].content

'Email sent successfully to azrapatvi1234@gmail.com'

In [15]:
for r in final_res['messages']:
    r.pretty_print()

================================ Human Message =================================


        My name is Azra Patvi.
        I am an AI and Data Science graduate.
        I have skills in Python and Machine Learning.
        I want to apply for an ML Engineer position at Cognizant.
        The recruiter's email id is azrapatvi1234@gmail.com
        
================================== Ai Message ==================================
================================== Ai Message ==================================

Email approved. Sending email...
================================== Ai Message ==================================

Email sent successfully to azrapatvi1234@gmail.com


In [16]:
config = {"configurable": {"thread_id": "chat_001"}}

while True:
    user_input = input("Bot: enter your details with recruiters email addres and all.\nYou: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    result = graph.invoke(
        {"messages": [HumanMessage(user_input)], "resume_text": resume_text},
        config=config
    )

    # If the graph is waiting for approval
    if "__interrupt__" in result:
        data = result["__interrupt__"][0].value
        print("\nSubject:", data["subject"])
        print(data["body"])
        answer = input("\nSend this email? (yes/no): ")

        result = graph.invoke(Command(resume={"approved": answer}), config=config)

    print("Bot:", result["messages"][-1].content)


Subject: Application for ML Engineer – Cognizant
Dear Hiring Manager,

I am writing to apply for the ML Engineer position at Cognizant. I am currently completing a B.Tech in Artificial Intelligence and Data Science and have hands‑on experience building end‑to‑end machine‑learning solutions. I am excited about the opportunity to contribute to Cognizant’s innovative AI initiatives.

Relevant Professional Experience:
**Data Scientist Intern – Vinayak IT Solutions (Feb 2026 – May 2026)**  
- Performed data preprocessing, exploratory analysis, feature engineering, model training and evaluation using Python and Scikit‑learn.  
- Developed a Walmart Sales Prediction System employing Linear Regression, Random Forest, Gradient Boosting, ARIMA, SARIMA and Prophet, achieving an R² of ≈0.93, and delivered it as an interactive Flask web app for real‑time forecasting.  
- Built a Customer Support Ticket Analysis & Satisfaction Prediction System to uncover complaint patterns and predict satisfaction